## TRAIN.CSV- EDA AND PREPROCESSING

## Planned model labels

### Civil Comments model
This model will predict:
- Broad toxicity
- Obscene/profane language
- Insults
- Threats
- Identity attacks
- Sexually explicit content

### Davidson model
The `class` column means:
- `0` = hate speech
- `1` = offensive language
- `2` = neither

For the broad abuse prediction, classes `0` and `1` will count as abusive, and class `2` as non-abusive. Davidson does not have separate labels for threats, insults, or profanity.

### Ensemble
The two models will contribute to a shared broad abuse prediction. The Civil Comments model will provide the more specific category predictions.

# EDA-CIVIL_DATASET


In [1]:
import pandas as pd

civil_df = pd.read_csv("civil_comments_sample.csv")
davidson_df = pd.read_csv("train.csv")

In [2]:
civil_df.shape

(50000, 8)

In [3]:
civil_df.describe()

,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,0.103082,0.004604,0.013997,0.009351,0.081054,0.022486,0.006496
std,0.196056,0.022900,0.065570,0.048949,0.174806,0.078174,0.044575
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.166667,0.000000,0.000000,0.000000,0.100000,0.000000,0.000000
max,1.000000,0.500000,1.000000,1.000000,1.000000,1.000000,0.909028


In [4]:
civil_df.head()

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,Socialists? What the hell does that have to do...,0.300000,0.000000,0.300000,0.0,0.200000,0.0,0.0
1,And I always thought that Eric Trump was the d...,0.808824,0.029412,0.088235,0.0,0.794118,0.0,0.0
2,I’m disappointed there’s no report from CB on ...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0
3,Nothing like subsidizing a country that is thr...,0.166667,0.000000,0.000000,0.0,0.166667,0.0,0.0
4,The only way to save the rhino is to legalize ...,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0


In [5]:
civil_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   text             50000 non-null  str    
 1   toxicity         50000 non-null  float64
 2   severe_toxicity  50000 non-null  float64
 3   obscene          50000 non-null  float64
 4   threat           50000 non-null  float64
 5   insult           50000 non-null  float64
 6   identity_attack  50000 non-null  float64
 7   sexual_explicit  50000 non-null  float64
dtypes: float64(7), str(1)
memory usage: 17.2 MB


In [6]:
civil_df.isnull().sum()

text               0
toxicity           0
severe_toxicity    0
obscene            0
threat             0
insult             0
identity_attack    0
sexual_explicit    0
dtype: int64

In [7]:
civil_df.duplicated().sum()

np.int64(66)

In [8]:
civil_df["text"].duplicated().sum()

np.int64(75)

In [9]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

repeated_text = civil_df[civil_df["text"].duplicated(keep=False)]

display(
    repeated_text[["text"] + label_columns]
    .sort_values("text")
    .head(50000)
)

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
6276,"""If you build it, they will come"".",0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
9955,"""If you build it, they will come"".",0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
16145,.,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
33569,.,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
25662,Agreed.,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...
20271,tgam test,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
22947,troll,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0
42510,troll,0.300000,0.0,0.0,0.0,0.300000,0.0,0.0
24185,‪Please sign petition for US President Trump t...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0


In [10]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

# Make a working copy and remove rows identical across every column
civil_clean = civil_df.drop_duplicates().copy()

print("Rows before:", len(civil_df))
print("Rows after exact-duplicate removal:", len(civil_clean))

# Check whether any repeated text has different labels
label_conflicts = (
    civil_clean.groupby("text")[label_columns]
    .nunique()
    .gt(1)
    .any(axis=1)
)

print("Repeated texts with different label scores:", label_conflicts.sum())

Rows before: 50000
Rows after exact-duplicate removal: 49934
Repeated texts with different label scores: 9


In [11]:
civil_clean = civil_clean.drop_duplicates(subset="text", keep="first").copy()

print("Rows after text deduplication:", len(civil_clean))
print("Repeated texts remaining:", civil_clean["text"].duplicated().sum())

Rows after text deduplication: 49925
Repeated texts remaining: 0


- We found out using `.duplicated().sum()` that some rows were duplicated. First, we checked how many rows there were, and we then dealt with it. 
- There were 66 rows that were duplicated, 9 repeated texts so we removed those duplicated rows and saved it in another copy. 
- Then we saw that Some duplicated rows after deduplicating were still left that had repeated texts with different label scores, so we removed them also. 

In [20]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

civil_exact_clean = civil_df.drop_duplicates().copy()

civil_clean = (
    civil_exact_clean
    .groupby("text", as_index=False)[label_columns]
    .mean()
)

print("Rows after removing exact duplicate rows:", len(civil_exact_clean))
print("Rows after grouping repeated text:", len(civil_clean))
print("Repeated texts remaining:", civil_clean["text"].duplicated().sum())

Rows after removing exact duplicate rows: 49934
Rows after grouping repeated text: 49925
Repeated texts remaining: 0


# EDA-DAVISON DATASET

In [12]:
davidson_df.head()


,count,hate_speech_count,offensive_language_count,neither_count,class,tweet
0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [13]:
davidson_df.shape


(24783, 6)

In [14]:
davidson_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 24783 entries, 0 to 24782
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   count                     24783 non-null  int64
 1   hate_speech_count         24783 non-null  int64
 2   offensive_language_count  24783 non-null  int64
 3   neither_count             24783 non-null  int64
 4   class                     24783 non-null  int64
 5   tweet                     24783 non-null  str  
dtypes: int64(5), str(1)
memory usage: 3.2 MB


In [15]:
davidson_df.isnull().sum()


count                       0
hate_speech_count           0
offensive_language_count    0
neither_count               0
class                       0
tweet                       0
dtype: int64

In [16]:
davidson_df.duplicated().sum()


np.int64(0)

In [17]:
davidson_df["class"].value_counts()

class
1    19190
2     4163
0     1430
Name: count, dtype: int64

In [18]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

print("CIVIL COMMENTS")
print("Rows:", len(civil_clean))
print("Missing values:")
print(civil_clean[["text"] + label_columns].isnull().sum())
print("Label score summaries:")
print(civil_clean[label_columns].describe().round(3))

print("\nDAVIDSON")
print("Rows:", len(davidson_df))
print("Missing values:")
print(davidson_df[["tweet", "class"]].isnull().sum())
print("Class counts (0=hate speech, 1=offensive, 2=neither):")
print(davidson_df["class"].value_counts().sort_index())

CIVIL COMMENTS
Rows: 49925
Missing values:
text               0
toxicity           0
severe_toxicity    0
obscene            0
threat             0
insult             0
identity_attack    0
sexual_explicit    0
dtype: int64
Label score summaries:
        toxicity  severe_toxicity    obscene     threat     insult  \
count  49925.000        49925.000  49925.000  49925.000  49925.000   
mean       0.103            0.005      0.014      0.009      0.081   
std        0.196            0.023      0.066      0.049      0.175   
min        0.000            0.000      0.000      0.000      0.000   
25%        0.000            0.000      0.000      0.000      0.000   
50%        0.000            0.000      0.000      0.000      0.000   
75%        0.167            0.000      0.000      0.000      0.100   
max        1.000            0.500      1.000      1.000      1.000   

       identity_attack  sexual_explicit  
count        49925.000        49925.000  
mean             0.023            0.00

## preprocessing

In [21]:
civil_model_df = civil_clean.copy()
davidson_model_df = davidson_df.copy()

civil_model_df["text"] = civil_model_df["text"].str.strip()
davidson_model_df["tweet"] = davidson_model_df["tweet"].str.strip()

civil_model_df = civil_model_df[civil_model_df["text"] != ""].copy()
davidson_model_df = davidson_model_df[davidson_model_df["tweet"] != ""].copy()

print("Civil rows:", len(civil_model_df))
print("Davidson rows:", len(davidson_model_df))

Civil rows: 49925
Davidson rows: 24783
